# agentMaxRuntime — from raw transcripts to the dev-card number

This notebook walks the **exact production pipeline** that produces this
metric's value on the profile page, starting from raw `~/.claude/projects/**/*.jsonl`
transcripts. Every step prints an interim value so you can see what the extractor
is doing.

**Formula (LaTeX):**

```
\text{max\_runtime} = \max_{s \in \text{sessions}} \text{afk\_max\_streak}(s)
```

**Parts:**

- **L** = `longest_interval_minutes` (min) — Length of the longest single contiguous AFK interval across the 30-day window. Currently undercounts subagent runtime (see docs/debugging/metric-verification.md § Known bugs).

**Server aggregator:** `server/web/lib/scorer/metrics/agentMaxRuntime.ts`
**Wire fields read:** `sessions[].afk_max_streak_minutes`


## Step 1 — Discover sessions

List every JSONL transcript under `~/.claude/projects/`.


In [ ]:
from pathlib import Path
import sys, os, json, uuid
# Make the `scripts/` package importable regardless of where the
# notebook is opened from. Walk upward from cwd until we find a
# directory containing `scripts/extractor.py` (the client root).
_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / 'scripts' / 'extractor.py').exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
    if _p.parent == _p:
        break
    _p = _p.parent
else:
    raise RuntimeError(
        'could not locate the client root (looking for scripts/extractor.py). '
        f'Started from {Path.cwd()}.'
    )
from scripts.events import find_sessions

sessions = find_sessions()
print(f'discovered {len(sessions)} JSONL transcripts')
for s in sessions[:5]:
    print(f'  {s.session_id[:12]}  project={s.project_root}  '
          f'first={s.first_ts_ms}  last={s.last_ts_ms}')


## Step 2 — Run the production extractor

`extract(device_id, client_version)` is the **single function** that powers
the production `/conductorscore` upload. It scans all sessions, classifies
minutes as HITL/AFK/Idle, counts tool invocations, detects plan signals,
etc., and returns an `ExtractorOutput` matching the wire format.


In [ ]:
from scripts.extractor import extract

payload = extract(device_id=str(uuid.UUID(int=0)), client_version='notebook')
wire = json.loads(payload.to_json())
print(f'extracted {len(wire["sessions"])} sessions into the wire payload')
print(f'schema_version: {wire["device"]["schema_version"]}')


## Step 3 — Inspect the wire fields this metric reads

Per the registry, this metric reads the following wire fields:

- `sessions[].afk_max_streak_minutes`

Below are the per-session values for those fields, plus a small distribution summary.


In [ ]:
WIRE_FIELDS = ["sessions[].afk_max_streak_minutes"]

def read(path: str, session: dict):
    if path.startswith('sessions[].'):
        return session.get(path[len('sessions[].'):])
    if path.startswith('config.'):
        return wire.get('config', {}).get(path[len('config.'):])
    return wire.get(path)

for field in WIRE_FIELDS:
    if field.startswith('config.'):
        print(f'{field}: {read(field, {})!r}')
        continue
    values = [read(field, s) for s in wire['sessions']]
    nonzero = [v for v in values if v not in (0, None, [], False)]
    print(f'{field}: {len(nonzero)} non-zero sessions, top 5: ' + str(
        sorted([v for v in nonzero if isinstance(v, (int, float))], reverse=True)[:5] or nonzero[:5]
    ))


### Inspector — AFK distribution

Top sessions by AFK max streak (in minutes). If your headline is surprisingly low,
this is where you'll see why — most sessions probably have very short AFK runs.


In [ ]:
afks = sorted([(s.get('afk_max_streak_minutes', 0), s.get('session_hash', '?')) for s in wire['sessions']], reverse=True)
print('Top 10 sessions by AFK max streak:')
for streak, sh in afks[:10]:
    print(f'  {sh[:12]}  streak={streak} min')
import statistics
vals = [a for a, _ in afks]
if vals:
    print(f'\nDistribution: min={min(vals)}  median={statistics.median(vals)}  max={max(vals)}  mean={statistics.mean(vals):.1f}')


## Step 4 — Apply the server aggregator

The TypeScript aggregator at `server/web/lib/scorer/metrics/agentMaxRuntime.ts` consumes the
wire payload via the same `aggregate-one.ts` adapter the debug script uses.
We shell out to it through `npx tsx` and parse the result. The output is
the canonical `{ raw, parts }` shape that the API surfaces and the dev-card
tile renders.


In [ ]:
import subprocess
from scripts.debug_metric.registry import _SERVER_WEB, _node20_path_env

result = subprocess.run(
    ['npx', 'tsx', 'scripts/aggregate-one.ts', 'agentMaxRuntime'],
    cwd=_SERVER_WEB,
    input=json.dumps(wire),
    capture_output=True, text=True, check=True,
    env=_node20_path_env(),
)
agg = json.loads(result.stdout)
print(json.dumps(agg, indent=2))


## Step 5 — Format like the dev-card

The profile tile renders the `raw` field with metric-specific formatting.
This cell mirrors `components/dev-card.tsx` exactly for `agentMaxRuntime` — the
final printed string matches what `/u/<your-handle>` displays for the
most recent upload.


In [ ]:
raw = agg.get('raw')
parts = agg.get('parts', [])

# Per-metric formatters mirror components/dev-card.tsx exactly.
# Each entry produces (headline, suffix) like the prototype's
# `<span class="val">{headline}<small>{suffix}</small></span>`.
def fmt_dev_card(metric_id, raw):
    if raw is None:
        return '—', ''
    if metric_id == 'humanAgentWallclock':
        # raw = {human: minutes, agent: minutes}; UI shows 'H / A h'.
        h = round(raw['human'] / 60); a = round(raw['agent'] / 60)
        return f'{h} / {a}', ' h'
    if metric_id == 'agentParallelism':
        return f'{raw:.2f}', '×'
    if metric_id == 'agentMaxRuntime':
        return f'{raw}', ' min'
    if metric_id == 'codingWithoutPlan':
        return f'{round(raw * 100)}', '% of significant edits'
    if metric_id == 'autoCompactionRate':
        return f'{raw:.1f}', ' per 100k tokens'
    if metric_id == 'revertRate':
        return f'{raw:.1f}', ' per coding session'
    if metric_id == 'repetitivePrompts':
        return f'{round(raw * 100)}', '% of long prompts'
    if metric_id == 'claudeMdBloat':
        return f'{raw}', ' lines'
    if metric_id == 'redundantApprovals':
        return f'{raw:.1f}', ' per session'
    if metric_id == 'rageQuit':
        return f'{round(raw * 100)}', ' per 100 sessions'
    if metric_id == 'topTierShare':
        # Scorer stores raw = 1 - max_tier_share. UI shows (1-raw)*100.
        return f'{round((1 - raw) * 100)}', '% on most expensive model'
    if metric_id == 'modelFreshness':
        return f'{round(raw * 100)}', '% on current ver.'
    if metric_id in ('mcpUsage', 'skillUsage', 'toolUsage', 'pluginUsage'):
        return f"{raw['invocations']} / {raw['distinct']}", ''
    if metric_id == 'costAggregate':
        if raw < 1: return '<$1', ''
        return f'${round(raw):,}', ''
    if metric_id == 'tokensAggregate':
        total = raw['total'] if isinstance(raw, dict) else raw
        if total >= 1_000_000: return f'{round(total / 1_000_000):,}M', ''
        if total >= 1_000:     return f'{round(total / 1_000):,}k', ''
        return f'{total:,}', ''
    return f'{raw}', ''

headline, suffix = fmt_dev_card('agentMaxRuntime', raw)
print(f'\n=== DEV-CARD VALUE ===')
print(f'{headline}{suffix}')
print(f'======================')
print()
print('Sub-metrics (shown in the (i) modal):')
for p in parts:
    print(f'  {p["symbol"]} = {p["value"]:,} {p["unit"]}')


## Where this lands on the page

The number printed by Step 5 is what the dev-card tile renders for `agentMaxRuntime`.
Open `https://conductorscore.com/u/<your-handle>` and find the tile matching
selector `[data-tile="agent-max-runtime"]`. Click the **ⓘ** button to see the same parts
breakdown printed above, with each symbol annotated by its `label` and `describe`.

For end-to-end provenance (UI ↔ API ↔ DB ↔ upload ↔ this re-extract), run:

```bash
python -m scripts.debug_metric agentMaxRuntime --user <your-handle>
```
